# Week 72, Day 1: Fixtures and parametrize

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w072d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import pytest

with open("test_warm.py", "w") as f:
    f.write('''
import pytest

LOG = []

@pytest.fixture
def resource():
    LOG.append("set up")
    yield "thing"
    LOG.append("tear down")

def test_one(resource):
    LOG.append("test one")

def test_two(resource):
    LOG.append("test two")

def test_log():
    print(LOG)
    assert LOG == ["set up", "test one", "tear down", "set up", "test two", "tear down"]
''')

pytest.main(["-q", "-p", "no:cacheprovider", "test_warm.py"])

In [ ]:
import pytest

with open("test_order.py", "w") as f:
    f.write('''
import pytest

LOG = []

@pytest.fixture
def db():
    LOG.append("db up")
    yield {"users": []}
    LOG.append("db down")

@pytest.fixture
def user(db):
    LOG.append("user up")
    db["users"].append("ada")
    yield "ada"
    LOG.append("user down")

def test_user(user, db):
    LOG.append(f"test sees {db['users']}")

def test_after():
    assert LOG == ["db up", "user up", "test sees ['ada']", "user down", "db down"]
''')

pytest.main(["-q", "-p", "no:cacheprovider", "test_order.py"])

In [ ]:
import pytest

with open("test_scope.py", "w") as f:
    f.write('''
import pytest

MADE = []

@pytest.fixture(scope="module")
def catalogue():
    MADE.append("catalogue")                 # pretend this is slow
    return ("tea", "jam", "bread")

@pytest.fixture
def basket(catalogue):
    MADE.append("basket")
    return []

def test_one(basket, catalogue):
    basket.append(catalogue[0])
    assert basket == ["tea"]

def test_two(basket, catalogue):
    assert basket == []                      # a fresh basket every time

def test_three(basket, catalogue):
    pass

def test_counts():
    assert MADE.count("catalogue") == 1 and MADE.count("basket") == 3
''')

pytest.main(["-q", "-p", "no:cacheprovider", "test_scope.py"])

In [ ]:
import pytest

with open("conftest.py", "w") as f:
    f.write('''
import random
import pytest

@pytest.fixture(autouse=True)
def fixed_seed():
    random.seed(42)                       # every test sees the same "random" numbers

@pytest.fixture
def make_order():
    made = []
    def make(*items, discount=0):
        order = {"items": list(items), "discount": discount}
        made.append(order)
        return order
    yield make
    print(f"made {len(made)} orders")
''')

with open("test_factory.py", "w") as f:
    f.write('''
import random

def test_two_orders(make_order):
    a = make_order("tea", "jam")
    b = make_order("bread", discount=10)
    assert len(a["items"]) == 2 and b["discount"] == 10

def test_seed_is_fixed():
    first = random.random()
    random.seed(42)
    assert random.random() == first
''')

pytest.main(["-q", "-s", "-p", "no:cacheprovider", "test_factory.py"])

In [ ]:
import pytest

with open("stacks.py", "w") as f:
    f.write('''
from collections import deque

class ListStack:
    def __init__(self): self._items = []
    def push(self, x): self._items.append(x)
    def pop(self): return self._items.pop()
    def __len__(self): return len(self._items)

class DequeStack:
    def __init__(self): self._items = deque()
    def push(self, x): self._items.append(x)
    def pop(self): return self._items.pop()
    def __len__(self): return len(self._items)
''')

with open("test_stacks.py", "w") as f:
    f.write('''
import pytest
from stacks import ListStack, DequeStack

@pytest.fixture(params=[ListStack, DequeStack], ids=lambda cls: cls.__name__)
def stack(request):
    return request.param()

def test_last_in_first_out(stack):
    stack.push(1); stack.push(2)
    assert stack.pop() == 2 and len(stack) == 1

@pytest.mark.parametrize("count", [0, 1, 5], ids=["none", "one", "five"])
def test_length(stack, count):
    for i in range(count):
        stack.push(i)
    assert len(stack) == count

@pytest.mark.parametrize("x", [1, 2])
@pytest.mark.parametrize("y", [10, 20, 30])
def test_pairs(x, y):
    assert x < y
''')

pytest.main(["-v", "-p", "no:cacheprovider", "test_stacks.py"])

> **Key idea.** Fixtures are requested by parameter name; yield fixtures clean up after the test, in reverse order of set-up, and each fixture runs at most once per test. Scopes (function, class, module, package, session) trade independence for speed: a fixture may only use fixtures of the same or wider scope, and shared fixtures must not be changed by tests. Factories return a function that makes objects and cleans them all up. conftest.py shares fixtures across files; autouse applies one everywhere. parametrize takes ids and pytest.param marks, stacks into every combination, and params= on a fixture runs whole suites against several implementations.

### Exercise: A fixture that cleans up

`resources.py` has a `Connection` class that logs `open`, `query` and `close` lines to `log.txt`. Write a **yield** fixture `conn` that opens `Connection("test")`, yields it and closes it afterwards, and at least two tests that use it to run a query (`conn.query("select 1")` returns `"SELECT 1"`).

In [ ]:
# Run this cell first: it creates the files used below.
with open('resources.py', 'w', encoding='utf-8') as f:
    f.write('def _log(line):\n    with open("log.txt", "a") as f:\n        f.write(line + "\\n")\n\nclass Connection:\n    def __init__(self, name):\n        self.name, self.closed = name, False\n        _log(f"open {self.name}")\n\n    def query(self, text):\n        if self.closed:\n            raise RuntimeError("connection is closed")\n        _log(f"query {self.name}")\n        return text.upper()\n\n    def close(self):\n        self.closed = True\n        _log(f"close {self.name}")\n')

In [ ]:
import pytest
from resources import Connection

@pytest.fixture
def conn():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cleans_up():\n    \"every test gets an open connection, closed afterwards\"\n    write_file(\"log.txt\", \"\")\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and len(r[\"passed\"]) >= 2, r[\"output\"][-600:]\n    log = read_file(\"log.txt\").splitlines()\n    assert log.count(\"open test\") >= 2 and log.count(\"open test\") == log.count(\"close test\"), log\n    assert all(log[i] == \"open test\" and log[i + 2] == \"close test\" for i in range(0, len(log), 3)), log")

### Exercise: One connection for the whole file

Opening a connection is slow. Using `resources.py` again, write a **module-scoped** fixture `shared` that opens `Connection("shared")` once and closes it after the file's last test, and at least **three** tests that each run a query on it.

In [ ]:
# Run this cell first: it creates the files used below.
with open('resources.py', 'w', encoding='utf-8') as f:
    f.write('def _log(line):\n    with open("log.txt", "a") as f:\n        f.write(line + "\\n")\n\nclass Connection:\n    def __init__(self, name):\n        self.name, self.closed = name, False\n        _log(f"open {self.name}")\n\n    def query(self, text):\n        if self.closed:\n            raise RuntimeError("connection is closed")\n        _log(f"query {self.name}")\n        return text.upper()\n\n    def close(self):\n        self.closed = True\n        _log(f"close {self.name}")\n')

In [ ]:
import pytest
from resources import Connection

@pytest.fixture
def shared():
    c = Connection("shared")
    yield c
    c.close()

def test_a(shared):
    assert shared.query("a") == "A"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_once():\n    \"one open, three or more queries, one close at the end\"\n    write_file(\"log.txt\", \"\")\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and len(r[\"passed\"]) >= 3, r[\"output\"][-600:]\n    log = read_file(\"log.txt\").splitlines()\n    assert log[0] == \"open shared\" and log[-1] == \"close shared\", log\n    assert log.count(\"open shared\") == 1 and log.count(\"query shared\") >= 3, log")

### Exercise: A fixture factory

`bank.py` has `Account(owner, balance=0)` with `deposit`, `withdraw` (raising `ValueError` if there isn't enough money) and `transfer(other, amount)`. Write a fixture **`make_account`** that returns a function making accounts, then tests using it that pass on the correct code and catch every broken version. You'll need two accounts at once for transfers.

In [ ]:
# Run this cell first: it creates the files used below.
with open('bank.py', 'w', encoding='utf-8') as f:
    f.write('class Account:\n    def __init__(self, owner, balance=0):\n        self.owner, self.balance = owner, balance\n\n    def deposit(self, amount):\n        if amount <= 0:\n            raise ValueError("amount must be positive")\n        self.balance += amount\n\n    def withdraw(self, amount):\n        if amount <= 0:\n            raise ValueError("amount must be positive")\n        if amount > self.balance:\n            raise ValueError("not enough money")\n        self.balance -= amount\n\n    def transfer(self, other, amount):\n        self.withdraw(amount)\n        other.deposit(amount)\n')

In [ ]:
import pytest
from bank import Account

@pytest.fixture
def make_account():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"bank.py\").read()\nBROKEN = {\n    \"withdraw allows overdraft\": GOOD.replace(\"if amount > self.balance:\", \"if amount > self.balance + 100:\"),\n    \"withdraw of exact balance refused\": GOOD.replace(\"if amount > self.balance:\", \"if amount >= self.balance:\"),\n    \"transfer doesn't take the money\": GOOD.replace(\"        self.withdraw(amount)\\n        other.deposit\", \"        other.deposit\"),\n    \"transfer pays the sender\": GOOD.replace(\"other.deposit(amount)\", \"self.deposit(amount)\"),\n    \"deposit of zero allowed\": GOOD.replace(\"    def deposit(self, amount):\\n        if amount <= 0:\", \"    def deposit(self, amount):\\n        if amount < 0:\"),\n}\ndef test_catches():\n    \"your tests pass on the correct code and catch all five broken versions\"\n    assert uses(\"make_account\"), \"use a fixture called make_account\"\n    results = against(source, \"bank\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"bank.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Cases with names

`slugs.py` has `slugify(title)`, which lower-cases a title, turns runs of spaces and hyphens into one hyphen, drops anything that isn't a letter, digit or hyphen, and strips hyphens from the ends. Write **one** parametrized test, `test_slugify`, with at least four cases and `ids` that include `"spaces"` and `"punctuation"`, which passes on the correct code and catches every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('slugs.py', 'w', encoding='utf-8') as f:
    f.write('import re\n\ndef slugify(title):\n    text = title.lower()\n    text = re.sub(r"[\\s-]+", "-", text)\n    text = re.sub(r"[^a-z0-9-]", "", text)\n    return text.strip("-")\n')

In [ ]:
import pytest
from slugs import slugify

@pytest.mark.parametrize("title, slug", [
    ("Hello", "hello"),
])
def test_slugify(title, slug):
    assert slugify(title) == slug

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"slugs.py\").read()\nBROKEN = {\n    \"keeps capitals\": GOOD.replace(\"title.lower()\", \"title\"),\n    \"one hyphen per space\": GOOD.replace(r\"[\\s-]+\", r\"[\\s-]\"),\n    \"keeps punctuation\": GOOD.replace('text = re.sub(r\"[^a-z0-9-]\", \"\", text)\\n', \"\"),\n    \"doesn't strip the ends\": GOOD.replace('text.strip(\"-\")', \"text\"),\n    \"drops digits\": GOOD.replace(\"a-z0-9-\", \"a-z-\"),\n}\ndef test_named_cases():\n    \"the cases have names, and catch all five broken versions\"\n    r = pytest_run(source)\n    names = r[\"passed\"]\n    assert r[\"failed\"] == [] and len(names) >= 4, r[\"output\"][-600:]\n    assert \"test_slugify[spaces]\" in names and \"test_slugify[punctuation]\" in names, names\n    results = against(source, \"slugs\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"slugs.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Exercise: Every combination

`units.py` converts between `"mm"`, `"cm"` and `"m"` with `convert(value, unit_from, unit_to)`. Write a test that stacks two `parametrize` decorators, one over `unit_from` and one over `unit_to` (all three units each, so **nine** combinations), and checks that converting `1` from one unit to the other and back gives `1` again (`pytest.approx`). Add one more test with known answers so that every broken version is caught.

In [ ]:
# Run this cell first: it creates the files used below.
with open('units.py', 'w', encoding='utf-8') as f:
    f.write('FACTORS = {"mm": 0.001, "cm": 0.01, "m": 1}\n\ndef convert(value, unit_from, unit_to):\n    return value * FACTORS[unit_from] / FACTORS[unit_to]\n')

In [ ]:
import pytest
from units import convert

UNITS = ["mm", "cm", "m"]

def test_round_trip():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"units.py\").read()\nBROKEN = {\n    \"upside down\": GOOD.replace(\"value * FACTORS[unit_from] / FACTORS[unit_to]\", \"value * FACTORS[unit_to] / FACTORS[unit_from]\"),\n    \"cm factor wrong\": GOOD.replace('\"cm\": 0.01', '\"cm\": 0.1'),\n    \"always returns the value\": GOOD.replace(\"return value * FACTORS[unit_from] / FACTORS[unit_to]\", \"return value\"),\n}\ndef test_stacked():\n    \"nine stacked cases and known answers\"\n    r = pytest_run(source)\n    trips = [n for n in r[\"passed\"] if n.startswith(\"test_round_trip[\")]\n    assert r[\"failed\"] == [] and len(trips) == 9, r[\"output\"][-600:]\n    results = against(source, \"units\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"units.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Exercise: One suite, two implementations

`queues.py` has two queue classes with the same interface: `ListQueue` and `LinkedQueue`, each with `put(x)`, `get()` (raising `IndexError` when empty) and `len()`. Write a **parametrized fixture** `queue` with `params=` over both classes, and tests using it, so that every test runs on both. Your tests must catch a bug in **either** class.

In [ ]:
# Run this cell first: it creates the files used below.
with open('queues.py', 'w', encoding='utf-8') as f:
    f.write('class ListQueue:\n    def __init__(self):\n        self._items = []\n\n    def put(self, x):\n        self._items.append(x)\n\n    def get(self):\n        if not self._items:\n            raise IndexError("get from an empty queue")\n        return self._items.pop(0)\n\n    def __len__(self):\n        return len(self._items)\n\n\nclass _Node:\n    def __init__(self, value):\n        self.value, self.next = value, None\n\n\nclass LinkedQueue:\n    def __init__(self):\n        self._head = self._tail = None\n        self._size = 0\n\n    def put(self, x):\n        node = _Node(x)\n        if self._tail is None:\n            self._head = self._tail = node\n        else:\n            self._tail.next = node\n            self._tail = node\n        self._size += 1\n\n    def get(self):\n        if self._head is None:\n            raise IndexError("get from an empty queue")\n        node = self._head\n        self._head = node.next\n        if self._head is None:\n            self._tail = None\n        self._size -= 1\n        return node.value\n\n    def __len__(self):\n        return self._size\n')

In [ ]:
import pytest
from queues import ListQueue, LinkedQueue

@pytest.fixture
def queue():
    return ListQueue()

def test_first_in_first_out(queue):
    queue.put(1)
    queue.put(2)
    assert queue.get() == 1

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"queues.py\").read()\nBROKEN = {\n    \"ListQueue is a stack\": GOOD.replace(\"return self._items.pop(0)\", \"return self._items.pop()\"),\n    \"LinkedQueue forgets the tail\": GOOD.replace(\"        if self._head is None:\\n            self._tail = None\\n\", \"\"),\n    \"LinkedQueue size wrong\": GOOD.replace(\"        self._size -= 1\\n\", \"\"),\n    \"LinkedQueue empty get returns None\": GOOD.replace('        if self._head is None:\\n            raise IndexError(\"get from an empty queue\")\\n', \"        if self._head is None:\\n            return None\\n\"),\n}\ndef test_both():\n    \"every test runs on both classes and catches a bug in either\"\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and len(r[\"passed\"]) >= 4, r[\"output\"][-600:]\n    assert any(\"Linked\" in n for n in r[\"passed\"]) and any(\"List\" in n for n in r[\"passed\"]), r[\"passed\"]\n    results = against(source, \"queues\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"queues.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Bug hunt: Closed before it was used

The fixture closes the connection and **returns** it, so every test gets a connection that's already closed, and both tests error. Make it a yield fixture, closing the connection after the test.

In [ ]:
# Run this cell first: it creates the files used below.
with open('resources.py', 'w', encoding='utf-8') as f:
    f.write('def _log(line):\n    with open("log.txt", "a") as f:\n        f.write(line + "\\n")\n\nclass Connection:\n    def __init__(self, name):\n        self.name, self.closed = name, False\n        _log(f"open {self.name}")\n\n    def query(self, text):\n        if self.closed:\n            raise RuntimeError("connection is closed")\n        _log(f"query {self.name}")\n        return text.upper()\n\n    def close(self):\n        self.closed = True\n        _log(f"close {self.name}")\n')

In [ ]:
import pytest
from resources import Connection

@pytest.fixture
def conn():
    c = Connection("test")
    c.close()
    return c

def test_one(conn):
    assert conn.query("one") == "ONE"

def test_two(conn):
    assert conn.query("two") == "TWO"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixed():\n    \"both tests pass and the connection is closed afterwards\"\n    write_file(\"log.txt\", \"\")\n    r = pytest_run(source)\n    assert sorted(r[\"passed\"]) == [\"test_one\", \"test_two\"] and r[\"failed\"] == [], r[\"output\"][-600:]\n    assert read_file(\"log.txt\").splitlines()[-1] == \"close test\"")

### Bug hunt: Tests that depend on their order

The `cart` fixture is module-scoped, so both tests share one list: `test_add` puts an item in it, and `test_empty_cart` (which runs second) fails. Give each test its own cart so both pass, in either order.

In [ ]:
import pytest

@pytest.fixture(scope="module")
def cart():
    return []

def test_add(cart):
    cart.append("tea")
    assert cart == ["tea"]

def test_empty_cart(cart):
    assert cart == []

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixed():\n    \"independent tests\"\n    r = pytest_run(source)\n    assert sorted(r[\"passed\"]) == [\"test_add\", \"test_empty_cart\"], r[\"output\"][-600:]\n    swapped = source.replace(\"def test_add\", \"def test_zz_add\")\n    r2 = pytest_run(swapped)\n    assert r2[\"failed\"] == [], r2[\"output\"][-600:]")

### Bug hunt: Names that don't match

`parametrize` names its arguments `"text, expected"`, but the test function's parameters are `word` and `expected`, so pytest stops with an error before running anything. Make the names agree so all three cases pass.

In [ ]:
import pytest

@pytest.mark.parametrize("text, expected", [
    ("level", True),
    ("rotor", True),
    ("hello", False),
])
def test_palindrome(word, expected):
    assert (word == word[::-1]) == expected

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixed():\n    \"three passing cases\"\n    r = pytest_run(source)\n    assert len(r[\"passed\"]) == 3 and r[\"failed\"] == [], r[\"output\"][-600:]")

### Puzzle: A fixture that always cleans up

Arrange the lines of a fixture that makes a temporary settings file, yields its path, and deletes it after the test. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # tests from @parametrize("a", [1, 2, 3]) stacked on @parametrize("b", [4, 5, 6, 7])
q = ___   # times a scope="module" fixture runs for a file of 5 tests that all use it
r = ___   # True or False: a scope="session" fixture may request a function-scoped fixture
s = ___   # set-up order "a b c" for a test using c, where c requests b and b requests a; the tear-down order
t = ___   # the attribute of request holding a parametrized fixture's current value

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 12 and q == 1 and r is False, \"p, q and r\"\n    assert s == \"c b a\" and t == \"param\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a tiny fixture system

Build the core of pytest's fixture machinery. `run_tests(tests, fixtures)`: `tests` is a list of test functions; `fixtures` maps a name to `(function, scope)`, with scope `"function"` or `"module"`. For each test, in order, work out its fixtures from its parameter names (`inspect.signature`), recursively (fixtures may request fixtures), set each up **at most once per test** (or once for the whole run if its scope is `"module"`), call the test with them, and record `(test name, "passed")`, or `"failed"` if it raised `AssertionError`, or `"error"` if it raised anything else. A fixture function that is a generator is a yield fixture: its value is what it yields, and after the test (or, for module scope, after **all** the tests) it's resumed to run its tear-down. Tear down in reverse order of set-up. A function fixture that requests a module fixture is fine; a module fixture requesting a function fixture raises `ValueError("scope mismatch")`. Return the list of results.

In [ ]:
import inspect

def run_tests(tests, fixtures):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fixture_system():\n    \"set-up, sharing, scopes and tear-down\"\n    log = []\n    def db():\n        log.append(\"db up\")\n        yield {\"rows\": []}\n        log.append(\"db down\")\n    def table(db):\n        log.append(\"table up\")\n        db[\"rows\"].append(\"t\")\n        yield \"t\"\n        log.append(\"table down\")\n    def number():\n        return 7\n    def test_a(table, db):\n        log.append(f\"a {db['rows']}\")\n    def test_b(number):\n        assert number == 8\n    def test_c(db):\n        log.append(f\"c {db['rows']}\")\n        raise KeyError\n    fixtures = {\"db\": (db, \"module\"), \"table\": (table, \"function\"), \"number\": (number, \"function\")}\n    assert run_tests([test_a, test_b, test_c], fixtures) == [(\"test_a\", \"passed\"), (\"test_b\", \"failed\"), (\"test_c\", \"error\")]\n    assert log == [\"db up\", \"table up\", \"a ['t']\", \"table down\", \"c ['t']\", \"db down\"]\n    log.clear()\n    fixtures[\"db\"] = (db, \"function\")\n    run_tests([test_a, test_c], fixtures)\n    assert log == [\"db up\", \"table up\", \"a ['t']\", \"table down\", \"db down\", \"db up\", \"c []\", \"db down\"]\n    def wide(number):\n        return number\n    def test_d(wide):\n        pass\n    try:\n        run_tests([test_d], {\"wide\": (wide, \"module\"), \"number\": (number, \"function\")})\n        assert False\n    except ValueError as e:\n        assert str(e) == \"scope mismatch\"")

## Warm-up

The fixture runs **once per test that asks for it**, and the code after `yield` runs straight after that test. `test_log` doesn't ask for the fixture, so nothing is set up for it. Today is about controlling exactly when that set-up and tear-down happen.

## Fixtures in depth

A fixture is a function marked `@pytest.fixture`. A test (or another fixture) **requests** it by naming it as a parameter; pytest finds the fixture with that name, calls it, and passes in the result. Week 24 Day 5 introduced them. The details that matter in a real suite:

- **`yield` fixtures** run their set-up, `yield` the value, and run their **tear-down** after the test, whether it passed or failed. If the set-up itself raises, the test is reported as an **error** (not a failure) and no tear-down runs, because nothing was set up.
- Fixtures can **request other fixtures**. pytest works out the order: a fixture's dependencies are set up first and torn down last, so tear-down always happens in the **reverse** order of set-up.
- A fixture is set up **at most once per test**, however many times it's requested in that test's tree of fixtures: if `test_x` asks for `db` and `user`, and `user` asks for `db` too, they share one `db`.
- **`request`** is a built-in fixture describing the test that asked: `request.node.name` is the test's name, `request.param` holds a parameter (below), and `request.addfinalizer(func)` registers clean-up, an alternative to `yield`.

## Scopes

By default a fixture has **function scope**: a fresh one for every test. That's what keeps tests independent. For expensive set-up (starting a server, loading a large file, creating a database schema) you can share one across many tests with `@pytest.fixture(scope=...)`:

| Scope | Created once per | Torn down |
|---|---|---|
| `"function"` (the default) | test | after that test |
| `"class"` | test class | after the class's last test |
| `"module"` | test file | after the file's last test |
| `"package"` | package of tests | after the package's last test |
| `"session"` | whole test run | at the very end |

Two rules follow:

- A fixture can only request fixtures of the **same or a wider** scope. A module fixture can't use a function fixture, because the function fixture would have to outlive its test; pytest raises a `ScopeMismatch` error.
- A shared fixture is **shared state**. If one test changes it, later tests see the change, and the tests start depending on their order. Share things that are slow to make and that tests only **read**, or pair a wide fixture with a narrow one that resets it: a session-scoped database, with a function-scoped fixture that opens a transaction and rolls it back after each test.

## Factories, conftest.py and autouse

**Fixture factories.** A fixture returns one value, but some tests need several objects, made with different arguments. The answer is a fixture that returns a **function** (a closure, from Week 60) that makes them, and can clean up everything it made:

```python
@pytest.fixture
def make_account():
    made = []
    def make(owner="ada", balance=0):
        account = Account(owner, balance)
        made.append(account)
        return account
    yield make
    for account in made:
        account.close()
```

A test then calls it as often as it likes: `a = make_account(balance=50)`, `b = make_account("bob")`.

**`conftest.py`.** Fixtures in a file called `conftest.py` are available to every test in that folder and below, with no import. A folder of tests can have its own `conftest.py` that adds fixtures, or **overrides** one from a parent folder by defining a fixture with the same name. Keep fixtures that one file uses in that file; move them to `conftest.py` when several files need them.

**`autouse=True`** makes a fixture apply to every test in its scope without being requested. It suits things every test needs and none should forget: resetting a global cache, fixing the random seed, blocking network access. Use it sparingly, because a test's parameters no longer show everything that affects it.

## Parametrize in depth

`@pytest.mark.parametrize(names, values)` runs one test once per set of values (Week 24 Day 2). The extras:

- **`ids=`** gives each case a readable name, shown in reports and usable with `-k`: `ids=["empty", "one", "many"]`, or a function that turns each value into a name.
- **`pytest.param(..., id="...", marks=...)`** sets a name or a mark for one case: `pytest.param(0, 0, marks=pytest.mark.xfail(reason="bug 12"))` expects that case to fail.
- **Stacking** two `parametrize` decorators runs **every combination**: 3 values times 4 values is 12 tests. It's the quickest way to test all pairs, and the quickest way to make a slow suite, so count before you stack.
- **Parametrized fixtures**: `@pytest.fixture(params=[...])` runs every test that uses the fixture once per parameter, with the value in `request.param`. The classic use is running a **whole suite** against several implementations of one interface, or several back ends (an in-memory store and a file store) with no test changed.

Read the names in the report: `test_length[ListStack-five]`, `test_pairs[10-1]` and so on. Two stacks times (1 + 3) tests, plus 6 pairs, is 14 tests from about 20 lines.

## Practice

These exercises use real pytest. **Check my answer** runs your test file with pytest and looks at what happened.

## On your own computer

Two command-line options make fixtures visible. Run them in any project with tests:

```bash
pytest --setup-show -q
```

```bash
pytest --fixtures-per-test -q
```

`--setup-show` prints every `SETUP` and `TEARDOWN` as it happens, with its scope, so you can see a module fixture made once and a function fixture made per test. `--fixtures-per-test` lists which fixtures each test uses, and where they're defined, which is how you find the `conftest.py` that a mystery fixture comes from.

## Quiz

## Daily challenge

## Before you go

You can now control exactly when every piece of test set-up happens, and run a whole suite over several implementations. Tomorrow: **mocking**, replacing the parts of a system that are slow, random or out of your control with stand-ins you can program and inspect.